# Procesador de Reportes Excel Filtrados

Este notebook automatiza la generación de archivos Excel individuales filtrados por una columna, preservando la estructura, estilos y fórmulas del archivo original. 

### Características:
- **Filtrado dinámico**: Genera un archivo por cada valor único de la columna de filtro.
- **Ocultación de hojas**: Permite ocultar hojas específicas de forma automática.
- **Protección de hojas**: Permite proteger hojas con contraseña.
- **Eficiencia máxima**: Todo el procesamiento se realiza en un solo paso por archivo.

In [ ]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook
import os
import shutil
import re

# ==================================================================
# CONFIGURACIÓN DE PARÁMETROS
# ==================================================================

# 1. Rutas de Archivos
INPUT_FILE = "/app/data/Plantilla_Matriz_Adición_Nivelacion_ V3_24032026_paradividir_correcion.xlsx"      # Nombre o ruta del archivo original
OUTPUT_DIR = "/app/data/salidas"        # Carpeta donde se guardarán los resultados

# 2. Estructura de la Hoja de Datos
SHEET_NAME = "ZONIFICACIÓN- PEDAGOGICO"             # Nombre de la hoja a procesar
HEADER_ROW = 3                   # Fila donde están los encabezados
DATA_START_ROW = 4               # Primera fila donde comienzan los datos reales

# 3. Filtro
FILTER_COL = "Regional UDS"           # Nombre exacto de la columna para filtrar

# 4. Ocultar Hojas (Opcional)
SHEETS_TO_HIDE = ["Parametros", "RP", "BASE_TASAS"]

# 5. Proteger Hojas (Opcional)
SHEETS_TO_PROTECT = ["ZONIFICACIÓN- PEDAGOGICO", "MATRIZ_CALCULADA"]           # Lista de hojas a proteger (ej: ["DATOS", "Resumen"])
PROTECTION_PASSWORD = "GMYC2026**"      # Contraseña para la protección

# ==================================================================

def clean_filename(filename):
    """Limpia el nombre de archivo para evitar caracteres no permitidos."""
    return re.sub(r'[\\/*?:"<>|]', "_", filename)

## 1. Cargar Datos y Verificar Valores Únicos
Cargamos con los parámetros definidos arriba.

In [ ]:
try:
    # pandas usa 0-indexing para cabeceras, por eso restamos 1
    df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, header=HEADER_ROW - 1)
    unique_values = df[FILTER_COL].unique()
    
    print(f"--- Datos cargados correctamente ---")
    print(f"Archivo: {INPUT_FILE}")
    print(f"Filas totales detectadas: {len(df)}")
    print(f"Valores únicos en '{FILTER_COL}': {len(unique_values)}")
    print(f"Lista: {unique_values}")
except Exception as e:
    print(f"Error al cargar {INPUT_FILE}: {e}")

## 2. Generar Reportes Individuales (Procesamiento Integral)
Copia la plantilla, filtra datos, oculta hojas y protege con contraseña en una sola operación por archivo.

In [ ]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Carpeta de salida creada: {OUTPUT_DIR}")

summary = []

for val in unique_values:
    group_name = "Sin_Nombre" if pd.isna(val) else str(val)
    safe_name = clean_filename(group_name)
    output_file = os.path.join(OUTPUT_DIR, f"Reporte_{safe_name}.xlsx")
    
    # Subconjunto de datos filtrados
    df_filtered = df[df[FILTER_COL] == val]
    
    # 1. Copiar el original como plantilla
    shutil.copy(INPUT_FILE, output_file)
    
    # 2. Abrir con openpyxl
    wb = load_workbook(output_file)
    ws_main = wb[SHEET_NAME]
    
    # 3. Limpiar datos anteriores (de DATA_START_ROW en adelante)
    max_row_template = ws_main.max_row
    if max_row_template >= DATA_START_ROW:
        ws_main.delete_rows(DATA_START_ROW, amount=max_row_template - DATA_START_ROW + 1)
        
    # 4. Escribir nuevos datos filtrados
    for r_idx, row in enumerate(df_filtered.values.tolist(), start=DATA_START_ROW):
        for c_idx, value in enumerate(row, start=1):
            ws_main.cell(row=r_idx, column=c_idx, value=value)
            
    # 5. Ocultar hojas especificadas
    hiddens = 0
    for sn in (SHEETS_TO_HIDE or []):
        if sn in wb.sheetnames:
            wb[sn].sheet_state = 'hidden'
            hiddens += 1

    # 6. Proteger hojas especificadas (Eficiencia: integrado en el ciclo)
    protected = 0
    for sn in (SHEETS_TO_PROTECT or []):
        if sn in wb.sheetnames:
            wb[sn].protection.set_password(PROTECTION_PASSWORD)
            protected += 1
    
    # 7. Guardar Cambios
    wb.save(output_file)
    wb.close()
    
    # Logging rápido
    status_msg = f"[Ok] {os.path.basename(output_file)} ({len(df_filtered)} filas)"
    if hiddens > 0: status_msg += f" | Hojas ocultas: {hiddens}"
    if protected > 0: status_msg += f" | Hojas protegidas: {protected}"
    print(status_msg)
    
    summary.append({"Grupo": group_name, "Archivo": output_file, "Filas": len(df_filtered), "Hojas_Ocultas": hiddens, "Protegidas": protected})

print("\n--- Proceso finalizado con éxito ---")

## 3. Resumen Final de Ejecución

In [ ]:
df_resumen = pd.DataFrame(summary)
if not df_resumen.empty:
    display(df_resumen)
else:
    print("No se generaron datos para el resumen.")